# NB-06-04 — Clustering, PCA and Anomaly Detection

**Module:** M06 · **Time:** 60–90 minutes · **Prerequisites:** L06-09

Segment customers, compress with PCA, and rank anomalies at a fixed alert budget. Nothing here
has a label until the last section, where the fraud labels are used only for evaluation.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

for parent in [Path.cwd(), *Path.cwd().parents]:
    candidate = parent / "modules" / "M06-machine-learning-fundamentals" / "code"
    if candidate.is_dir():
        sys.path.insert(0, str(candidate))
        break
print("code directory:", candidate)

In [ ]:
import numpy as np
import datasets as ds
from prep import StandardScaler
from unsupervised import (KMeans, AgglomerativeClustering, DBSCAN, GaussianMixture,
                          PCA, silhouette_score, zscore_anomalies, iqr_anomalies,
                          mahalanobis_scores)

## 2. k-means and the choice of k

In [ ]:
frame = ds.customers()
cols = ["annual_income_k", "spending_score", "recency_days", "purchase_frequency"]
X = StandardScaler().fit_transform(frame[cols].to_numpy())
for k in range(2, 7):
    km = KMeans(k, seed=0).fit(X)
    print(f"k={k} inertia {km.inertia_:8.1f} silhouette {silhouette_score(X, km.labels_):.3f}")

Silhouette prefers k=2; the planted segments are four. Profile the four-cluster
solution and judge for yourself.

In [ ]:
km = KMeans(4, seed=0).fit(X)
print("sizes:", np.bincount(km.labels_).tolist())
for k in range(4):
    profile = frame[cols].to_numpy()[km.labels_ == k].mean(axis=0)
    print(f"c{k}: " + "  ".join(f"{n}={v:.1f}" for n, v in zip(cols, profile)))

## 3. Hierarchical and density-based alternatives

In [ ]:
print("hierarchical, 4 clusters (150 rows):",
      np.bincount(AgglomerativeClustering(4, "average").fit(X[:150]).labels_).tolist())
for eps in [0.4, 0.5, 0.8]:
    labels = DBSCAN(eps, 5).fit(X).labels_
    print(f"eps={eps}: noise {int((labels == -1).sum())}, clusters {len(set(labels)) - (1 if -1 in labels else 0)}")

## 4. A Gaussian mixture and BIC

In [ ]:
for k in [2, 3, 4, 5]:
    gm = GaussianMixture(k, seed=0).fit(X)
    print(f"k={k} BIC {gm.bic(X):8.1f} loglik {gm.log_likelihood_history_[-1]:8.1f}")
gm = GaussianMixture(4, seed=0).fit(X)
resp = gm.predict_proba(X)
print("least confident customer:", int(np.argmin(resp.max(axis=1))),
      "membership:", np.round(resp[np.argmin(resp.max(axis=1))], 3))

## 5. PCA

In [ ]:
pca = PCA().fit(X)
print("ratios:", np.round(pca.explained_variance_ratio_, 3))
print("2 components keep:", round(float(pca.explained_variance_ratio_[:2].sum()), 3))
Z = pca.transform(X)
recon = pca.inverse_transform(Z[:, :2])
print("reconstruction MSE (2 comps):", round(float(np.mean((X - recon) ** 2)), 4))

## 6. Anomalies at a fixed budget

In [ ]:
tx = ds.transactions()
tcols = ["amount", "distance_from_home_km", "ratio_to_median_spend", "attempts_last_hour"]
Xt = StandardScaler().fit_transform(np.log1p(tx[tcols].to_numpy()))

flag = zscore_anomalies(Xt, 3.0)
print("z>3:", int(flag.sum()), "precision", round(tx.loc[flag, "is_fraud"].mean(), 3),
      "recall", round(tx.loc[flag, "is_fraud"].sum() / tx["is_fraud"].sum(), 3))

flag = iqr_anomalies(tx[tcols].to_numpy(), 1.5)
print("IQR:", int(flag.sum()), "precision", round(tx.loc[flag, "is_fraud"].mean(), 3))

scores = mahalanobis_scores(Xt)
for budget in [25, 50, 100]:
    idx = np.argsort(-scores)[:budget]
    print(budget, "precision", round(tx.iloc[idx]["is_fraud"].mean(), 3),
          "recall", round(tx.iloc[idx]["is_fraud"].sum() / tx["is_fraud"].sum(), 3))

The IQR rule catches every fraud and produces 932 alerts. The review team cannot use
that. Precision at the alert budget is the metric that matches the process.

## 7. Reflect

1. Which k would you present to marketing, and what would you say about the silhouette?
2. Why does the first principal component explain 87% of the variance here?
3. Would you ship the z-score, IQR or Mahalanobis detector? State the budget you assumed.

Next: the labs, then **QUIZ-06**.